In [1]:
import random
import torch
import os
import numpy as np
import pandas as pd
import polars as pl

In [2]:
INPUT_DIR = '.'

In [3]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [4]:
def make_aggregated_outputs(input_file, metrics = ['accuracy', 'precision', 'recall', 'f1', 'kappa', 'MCC']):
    dimensions = [
        'informational_vs_involved',
        'non-narrative_vs_narrative',
        'situation-dependent_vs_explicit',
        'non-persuasive_vs_persuasive',
        'non-abstract_vs_abstract',
        'compressed_vs_elaborated'
    ]

    saved_output_dict = {}

    for folder in os.listdir(INPUT_DIR):
        folder_path = os.path.join(INPUT_DIR, folder)

        if not (os.path.isdir(folder_path) and 'outputs' in folder):
            continue

        saved_output_dict[folder] = {
            dim: {metric: [] for metric in metrics}
            for dim in dimensions
        }

        for sub_folder in os.listdir(folder_path):
            sub_path = os.path.join(folder_path, sub_folder)

            if not (os.path.isdir(sub_path) and sub_folder.isdigit()):
                continue

            file_path = os.path.join(sub_path, f"{input_file}.csv")
            df = pd.read_csv(file_path)

            for dimension in dimensions:
                temp_df = df[df['dimension'] == dimension]

                if temp_df.empty:
                    raise ValueError(f"There should be something in the temp_df for the dimension {dimension}.")

                row = temp_df.iloc[0]

                for metric in metrics:
                    saved_output_dict[folder][dimension][metric].append(float(row[metric]))

    temp_dict_all = {}

    for folder, dimensions_dict in saved_output_dict.items():
        series_list = []

        for dimension, metrics_dict in dimensions_dict.items():
            mean_series = pd.Series({
                metric: (sum(values) / len(values)) if values else float('nan')
                for metric, values in metrics_dict.items()
            }, name=dimension)

            series_list.append(mean_series)

        df_folder = pd.concat(series_list, axis=1)
        temp_dict_all[folder] = df_folder

    df_all_folders = pd.concat(temp_dict_all, axis=0)

    df_mean_all = df_all_folders.groupby(level=1).mean()

    return df_all_folders, df_mean_all

In [5]:
all_classif, mean_classif = make_aggregated_outputs('classification_comparison_results_zero_vs_biber')

In [6]:
all_classif

informational_vs_involved  non-narrative_vs_narrative  \
outputsTrain accuracy                    0.601100                    0.553800   
             precision                   0.513943                    0.458026   
             recall                      0.671601                    0.595646   
             f1                          0.581261                    0.516920   
             kappa                       0.213503                    0.115869   
             MCC                         0.221195                    0.119510   
outputsTest  accuracy                    0.596500                    0.536200   
             precision                   0.507804                    0.438370   
             recall                      0.653448                    0.581826   
             f1                          0.570881                    0.499086   
             kappa                       0.200932                    0.083131   
             MCC                         0.207198                    0.086048   
outputsAll   accuracy                    0.596400                    0.536700   
             precision                   0.510188                    0.441725   
             recall                      0.661882                    0.581747   
             f1                          0.575247                    0.500989   
             kappa                       0.203190                    0.084147   
             MCC                         0.210113                    0.087168   

                        situation-dependent_vs_explicit  \
outputsTrain accuracy                          0.587500   
             precision                         0.640850   
             recall                            0.653447   
             f1                                0.646177   
             kappa                             0.151296   
             MCC                               0.151896   
outputsTest  accuracy                          0.580300   
             precision                         0.636582   
             recall                            0.643127   
             f1                                0.638842   
             kappa                             0.137317   
             MCC                               0.137995   
outputsAll   accuracy                          0.581200   
             precision                         0.636471   
             recall                            0.641919   
             f1                                0.638365   
             kappa                             0.140089   
             MCC                               0.140453   

                        non-persuasive_vs_persuasive  \
outputsTrain accuracy                       0.469300   
             precision                      0.342848   
             recall                         0.555922   
             f1                             0.423362   
             kappa                         -0.019591   
             MCC                           -0.021646   
outputsTest  accuracy                       0.474500   
             precision                      0.351152   
             recall                         0.561267   
             f1                             0.431226   
             kappa                         -0.010843   
             MCC                           -0.012473   
outputsAll   accuracy                       0.463700   
             precision                      0.335975   
             recall                         0.551669   
             f1                             0.416783   
             kappa                         -0.027869   
             MCC                           -0.030665   

                        non-abstract_vs_abstract  compressed_vs_elaborated  
outputsTrain accuracy                   0.538700                  0.528700  
             precision                  0.255896                  0.157692  
             recall                     0.595924                  

In [7]:
mean_classif

,informational_vs_involved,non-narrative_vs_narrative,situation-dependent_vs_explicit,non-persuasive_vs_persuasive,non-abstract_vs_abstract,compressed_vs_elaborated
MCC,0.212835,0.097576,0.143448,-0.021594,0.096463,-0.060715
accuracy,0.598000,0.542233,0.583000,0.469167,0.538633,0.529933
f1,0.575796,0.505665,0.641128,0.423790,0.356192,0.217761
kappa,0.205875,0.094382,0.142901,-0.019434,0.079281,-0.051677
precision,0.510645,0.446040,0.637968,0.343325,0.256137,0.158956
recall,0.662311,0.586406,0.646164,0.556286,0.593591,0.352009


In [8]:
all_contin, mean_contin = make_aggregated_outputs('continuous_comparison_results_zero_vs_biber', ['pearson', 'spearman', 'MSE', 'RMSE', 'MAE'])

In [9]:
all_contin

informational_vs_involved  non-narrative_vs_narrative  \
outputsTrain pearson                    0.288811                    0.127199   
             spearman                   0.298108                    0.188435   
             MSE                        1.422378                    1.745601   
             RMSE                       1.190498                    1.319679   
             MAE                        0.952394                    1.029810   
outputsTest  pearson                    0.267926                    0.104483   
             spearman                   0.278434                    0.159463   
             MSE                        1.464147                    1.791035   
             RMSE                       1.208494                    1.336838   
             MAE                        0.969567                    1.049829   
outputsAll   pearson                    0.267550                    0.096130   
             spearman                   0.275804                    0.158089   
             MSE                        1.464901                    1.807741   
             RMSE                       1.208431                    1.342875   
             MAE                        0.969493                    1.051650   

                       situation-dependent_vs_explicit  \
outputsTrain pearson                          0.087237   
             spearman                         0.152053   
             MSE                              1.825526   
             RMSE                             1.348866   
             MAE                              0.994517   
outputsTest  pearson                          0.067699   
             spearman                         0.143885   
             MSE                              1.864601   
             RMSE                             1.363196   
             MAE                              0.997671   
outputsAll   pearson                          0.115208   
             spearman                         0.166564   
             MSE                              1.769585   
             RMSE                             1.327820   
             MAE                              0.980434   

                       non-persuasive_vs_persuasive  non-abstract_vs_abstract  \
outputsTrain pearson                       0.011054                  0.083714   
             spearman                      0.032562                  0.151253   
             MSE                           1.977892                  1.832572   
             RMSE                          1.404795                  1.351187   
             MAE                           1.098227                  1.033019   
outputsTest  pearson                       0.023117                  0.091869   
             spearman                      0.027348                  0.150231   
             MSE                           1.953766                  1.816262   
             RMSE                          1.396049                  1.345839   
             MAE                           1.095911                  1.034969   
outputsAll   pearson                       0.015296                  0.089598   
             spearman                      0.020284                  0.163841   
             MSE                           1.969408                  1.820805   
             RMSE                          1.401507                  1.347130   
             MAE                           1.102655                  1.026743   

                       compressed_vs_elaborated  
outputsTrain pearson                  -0.014725  
             spearman                 -0.135515  
             MSE                       2.029450  
             RMSE                      1.422907  
             MAE                       1.003329  
outputsTest  pearson                  -0.024881  
             spearman                 -0.129901  
             MSE                       2.049762  
             RMSE                      1.430197  
             MAE

In [10]:
mean_contin

,informational_vs_involved,non-narrative_vs_narrative,situation-dependent_vs_explicit,non-persuasive_vs_persuasive,non-abstract_vs_abstract,compressed_vs_elaborated
MAE,0.963818,1.043763,0.990874,1.098931,1.031577,1.004054
MSE,1.450475,1.781459,1.819904,1.967022,1.823213,2.032659
RMSE,1.202474,1.333131,1.346627,1.400784,1.348052,1.423971
pearson,0.274762,0.109271,0.090048,0.016489,0.088393,-0.016329
spearman,0.284115,0.168662,0.154167,0.026732,0.155109,-0.128190
